In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"CUDA Version (PyTorch): {torch.version.cuda}")
    print(f"GPU Name:               {torch.cuda.get_device_name(0)}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM Available:   {total_memory:.2f} GB")
else:
    print("WARNING: CUDA not detected. Check your PyTorch installation.")

Using device: cuda
CUDA Version (PyTorch): 12.8
GPU Name:               NVIDIA A100 80GB PCIe
Total VRAM Available:   79.25 GB


In [2]:
import os
import urllib.request

# Create a data directory outside the repo
data_dir = '../data'
os.makedirs(data_dir, exist_ok=True)

base_url = "https://portal.nersc.gov/cfs/m4392/G25/"

files_to_download = [
    "Dataset_Specific_Unlabelled.h5",
    "Dataset_Specific_labelled_full_only_for_2i.h5"
]

for filename in files_to_download:
    url = base_url + filename
    filepath = os.path.join(data_dir, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename} from NERSC... (This is a large file, it may take a few minutes)")
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"Successfully downloaded {filename}!")
        except Exception as e:
            print(f"Error downloading {filename}: {e}")
    else:
        print(f"{filename} already exists. Skipping download.")

Dataset_Specific_Unlabelled.h5 already exists. Skipping download.
Dataset_Specific_labelled_full_only_for_2i.h5 already exists. Skipping download.


In [3]:
class CMSParticleDataset(Dataset):
    def __init__(self, file_path, is_labelled=True):
        self.file_path = file_path
        self.is_labelled = is_labelled
        
        with h5py.File(self.file_path, 'r') as f:
            # 1. Determine the image key 
            if 'X_jet' in f:
                self.img_key = 'X_jet'
            elif 'X' in f:
                self.img_key = 'X'
            elif 'jet' in f:
                self.img_key = 'jet'
            else:
                raise KeyError(f"No valid image key found. Keys: {list(f.keys())}")

            # 2. Get the length 
            self.length = len(f[self.img_key])
            
            # 3. Setup labels and mass keys matching the actual file structure
            if self.is_labelled:
                self.y_key = 'Y' # Capital Y!
                self.m_key = 'm' # Just 'm'

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        with h5py.File(self.file_path, 'r') as f:
            
            img_np = np.array(f[self.img_key][idx], dtype=np.float32)
            
            # --- PHYSICS-INFORMED PREPROCESSING ---
            # 1. Noise Suppression (Thresholding)
            img_np[img_np < 1e-3] = 0.0
            
            # 2. Log-Scaling to prevent high-energy pixel saturation
            img_np = np.log1p(img_np)
            
            # 3. Z-score Normalization (Standardization per channel)
            mean = img_np.mean(axis=(0, 1), keepdims=True)
            std = img_np.std(axis=(0, 1), keepdims=True) + 1e-6
            img_np = (img_np - mean) / std
            # --------------------------------------

            # Ensure correct channel dimension (Channels first)
            if img_np.shape[-1] == 8 or img_np.shape[-1] <= 4:
                img_np = np.transpose(img_np, (2, 0, 1))
                
            img = torch.tensor(img_np)
            
            # 2. Label and Mass extraction
            if self.is_labelled:
                label_data = np.array(f[self.y_key][idx])
                
                # Handle both one-hot and flat binary labels
                if label_data.size > 1:
                    label = torch.tensor(np.argmax(label_data), dtype=torch.long)
                else:
                    label = torch.tensor(int(label_data.item()), dtype=torch.long)

                mass = torch.tensor(f[self.m_key][idx], dtype=torch.float32)
                return img, label, mass
            
            return img

# --- Verification ---
unlabelled_path = '../data/Dataset_Specific_Unlabelled.h5'
if os.path.exists(unlabelled_path):
    print(f"Keys in file: {list(h5py.File(unlabelled_path, 'r').keys())}")
    dummy_dataset = CMSParticleDataset(unlabelled_path, is_labelled=False)
    print(f"Dataset size: {len(dummy_dataset)}")
    print(f"Sample shape: {dummy_dataset[0].shape} (Target: [8, 125, 125])")
else:
    print("Dataset not found. Check the path in Cell 2.")

Keys in file: ['jet']
Dataset size: 60000
Sample shape: torch.Size([8, 125, 125]) (Target: [8, 125, 125])


In [4]:
# --- Cell 4: DataLoader and Patch Embedding ---

# 1. Create the DataLoader for the unlabelled data
batch_size = 256 # You can push this higher (256, 512) on the A100 if you want
unlabelled_loader = DataLoader(
    dummy_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=4, # Speeds up data loading
    pin_memory=True # Speeds up CPU to GPU transfer
)

# Grab a single batch to test our layers
test_images = next(iter(unlabelled_loader))
print(f"Batch shape: {test_images.shape} -> (Batch, Channels, Height, Width)")

# 2. Build the Patch Embedding Layer
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=8, patch_size=5, embed_dim=256, img_size=125):
        super().__init__()
        self.patch_size = patch_size
        
        # Calculate how many patches we will have
        self.num_patches = (img_size // patch_size) ** 2
        
        # Use a 2D Convolution to extract patches and project them to embed_dim simultaneously
        self.proj = nn.Conv2d(
            in_channels, 
            embed_dim, 
            kernel_size=patch_size, 
            stride=patch_size
        )
        
        # Positional embeddings to tell the transformer where each patch came from
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, embed_dim))

    def forward(self, x):
        # x shape: (B, C, H, W) -> (128, 8, 125, 125)
        x = self.proj(x) 
        # New shape: (B, embed_dim, H/patch_size, W/patch_size) -> (128, 256, 25, 25)
        
        # Flatten the spatial dimensions: (B, embed_dim, num_patches) -> (128, 256, 625)
        x = x.flatten(2)
        
        # Swap dimensions for the transformer: (B, num_patches, embed_dim) -> (128, 625, 256)
        x = x.transpose(1, 2)
        
        # Add positional embeddings
        x = x + self.pos_embed
        return x

# 3. Test the Patch Embedding
embed_dim = 128 # Size of the hidden vectors in the transformer
patch_embedder = PatchEmbedding(in_channels=8, patch_size=5, embed_dim=embed_dim, img_size=125)

# Move to GPU and test
patch_embedder = patch_embedder.to(device)
test_images = test_images.to(device)

embedded_patches = patch_embedder(test_images)
print(f"Embedded shape: {embedded_patches.shape} -> (Batch, Num_Patches, Embed_Dim)")

Batch shape: torch.Size([256, 8, 125, 125]) -> (Batch, Channels, Height, Width)
Embedded shape: torch.Size([256, 625, 128]) -> (Batch, Num_Patches, Embed_Dim)


In [5]:
# --- Cell 5: Linear Attention Mechanism ---

class LinearAttention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.eps = 1e-6

    def forward(self, x):
        B, N, C = x.shape
        # qkv shape: [3, B, num_heads, N, head_dim]
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # ELU + 1 activation acts as the feature map for Linear Attention
        q = torch.nn.functional.elu(q) + 1
        k = torch.nn.functional.elu(k) + 1

        # Compute the context (K_T * V) first: [B, H, head_dim, head_dim]
        # This is the O(N) trick!
        context = torch.matmul(k.transpose(-2, -1), v)
        
        # Compute the normalizer (denominator)
        k_sum = k.sum(dim=-2) # [B, H, head_dim]
        den = torch.matmul(q, k_sum.unsqueeze(-1)) + self.eps
        
        # Apply context to Queries
        num = torch.matmul(q, context)
        x = num / den
        
        # Reshape and project back to original dim
        x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, dropout=0.1): # --- FIXED: Added dropout param ---
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = LinearAttention(dim, num_heads=num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout), # --- FIXED: Added Dropout ---
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout)  # --- FIXED: Added Dropout ---
        )
        self.drop = nn.Dropout(dropout) # --- FIXED: Added Dropout for attention ---

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm1(x)))
        x = x + self.mlp(self.norm2(x))
        return x

In [6]:
# --- Cell 6: The Full Linear ViT Model ---

class LinearViT(nn.Module):
    def __init__(self, in_channels=8, patch_size=5, img_size=125, embed_dim=128, depth=4, num_heads=8, num_classes=2):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim, img_size)
        
        # Sequence of Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads) for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        
        # Dual Heads: Classification and Regression
        self.cls_head = nn.Linear(embed_dim, num_classes)
        self.reg_head = nn.Linear(embed_dim, 1)

    def forward(self, x):
        # 1. Patchify and Embed
        x = self.patch_embed(x) # [B, 625, 128]
        
        # 2. Pass through Transformer Blocks
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        
        # 3. Global Average Pooling (condense 625 patches into 1 vector)
        x = x.mean(dim=1) # [B, 128]
        
        # 4. Final outputs
        logits = self.cls_head(x) # For classification
        mass = self.reg_head(x)   # For mass regression
        
        return logits, mass.squeeze(-1)

# Initialize the model
model = LinearViT().to(device)

# Test a forward pass with the batch from Cell 4
logits, mass = model(test_images)
print(f"Logits shape: {logits.shape} (Classification)")
print(f"Mass shape:   {mass.shape} (Regression)")

Logits shape: torch.Size([256, 2]) (Classification)
Mass shape:   torch.Size([256]) (Regression)


In [7]:
# --- Cell 7: Pre-training (Self-Supervised Reconstruction) ---

class ReconstructionDecoder(nn.Module):
    def __init__(self, embed_dim=128, out_channels=8):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 8 * 125 * 125), # Project back to full pixel space
            nn.Sigmoid() # Keeps pixel values between 0 and 1
        )

    def forward(self, x):
        x = self.decoder(x)
        return x.view(-1, 8, 125, 125)

# Initialize
decoder = ReconstructionDecoder().to(device)
optimizer = torch.optim.AdamW(list(model.parameters()) + list(decoder.parameters()), lr=1e-4)
criterion = nn.MSELoss()

# Pre-training Loop
epochs = 50
model.train()

print("Starting Pre-training on Unlabelled Data...")
for epoch in range(epochs):
    pbar = tqdm(unlabelled_loader, desc=f"Epoch {epoch+1}")
    epoch_loss = 0
    for imgs in pbar:
        imgs = imgs.to(device)
               
        # Manually run the transformer part:
        x = model.patch_embed(imgs)
        for block in model.blocks:
            x = block(x)
        latent = model.norm(x).mean(dim=1)
        
        recon_imgs = decoder(latent)
        loss = criterion(recon_imgs, imgs)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix(mse=loss.item())

print("Pre-training complete. Saving weights...")
torch.save(model.state_dict(), 'pretrained_vit.pth')

Starting Pre-training on Unlabelled Data...


Epoch 50: 100%|██████████| 235/235 [01:22<00:00,  2.85it/s, mse=0.966]

Pre-training complete. Saving weights...


In [8]:
# Run this in a new cell to check your data range
print(f"Max value in batch: {test_images.max().item()}")
print(f"Min value in batch: {test_images.min().item()}")

Max value in batch: 124.99431610107422
Min value in batch: -0.32149553298950195


In [9]:
# --- Cell 8: Labelled Data Loading & Split (Fixed) ---
from torch.utils.data import random_split # <--- This is the missing piece!

labelled_path = '../data/Dataset_Specific_labelled_full_only_for_2i.h5'

if os.path.exists(labelled_path):
    # Load the labelled dataset
    full_labelled_ds = CMSParticleDataset(labelled_path, is_labelled=True)
    
    # 80/20 Split
    train_size = int(0.8 * len(full_labelled_ds))
    val_size = len(full_labelled_ds) - train_size
    
    # Using a generator for reproducibility
    train_ds, val_ds = random_split(
        full_labelled_ds, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    # Dataloaders
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
    
    print(f"Labelled Data Ready.")
    print(f"Training samples: {len(train_ds)} | Validation samples: {len(val_ds)}")
else:
    print("Labelled dataset not found!")

Labelled Data Ready.
Training samples: 8000 | Validation samples: 2000


In [10]:
# --- Cell 9: Final Training Loop (Scratch vs. Finetuned) ---

def train_and_eval(model, mode_name, epochs=20, is_finetuning=False):
    print(f"\nStarting Training: {mode_name}")
    
    # --- DIFFERENTIAL LEARNING RATES ---
    if is_finetuning:
        # Protect the pre-trained backbone, train the new heads faster
        optimizer = torch.optim.AdamW([
            {'params': model.patch_embed.parameters(), 'lr': 1e-5},
            {'params': model.blocks.parameters(), 'lr': 1e-5},
            {'params': model.norm.parameters(), 'lr': 1e-5},
            {'params': model.cls_head.parameters(), 'lr': 1e-4},
            {'params': model.reg_head.parameters(), 'lr': 1e-4}
        ], weight_decay=0.01)
    else:
        # Scratch model learns everything at the same standard rate
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    # -----------------------------------
    
    cls_criterion = nn.CrossEntropyLoss()
    reg_criterion = nn.MSELoss()
    
    history = {'val_acc': [], 'val_mse': []}

    for epoch in range(epochs):
        model.train()
        for imgs, labels, masses in tqdm(train_loader, desc=f"{mode_name} - Epoch {epoch+1}"):
            imgs, labels, masses = imgs.to(device), labels.to(device), masses.to(device)
            
            logits, pred_mass = model(imgs)
            
            # Scale the mass down by 100 to stabilize MSE
            target_mass = masses.view(-1) / 100.0
            
            loss = cls_criterion(logits, labels) + reg_criterion(pred_mass, target_mass)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_loss, correct, total_mse = 0, 0, 0
        with torch.no_grad():
            for imgs, labels, masses in val_loader:
                imgs, labels, masses = imgs.to(device), labels.to(device), masses.to(device)
                logits, pred_mass = model(imgs)
                
                preds = logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
                
                # Apply the same scaling during validation evaluation
                target_mass = masses.view(-1) / 100.0
                total_mse += reg_criterion(pred_mass, target_mass).item()

        acc = correct / len(val_ds)
        avg_mse = total_mse / len(val_loader)
        print(f"[{mode_name}] Val Acc: {acc:.2%} | Val MSE (Scaled): {avg_mse:.4f}")
        history['val_acc'].append(acc)
        history['val_mse'].append(avg_mse)
        
    return history

# 1. Train from Scratch
print("--- Baseline: Training from Scratch ---")
model_scratch = LinearViT().to(device)
history_scratch = train_and_eval(model_scratch, "Scratch", epochs=20, is_finetuning=False)

# 2. Train with Finetuning
print("\n--- Experiment: Finetuning Pre-trained Model ---")
model_ft = LinearViT().to(device)
model_ft.load_state_dict(torch.load('pretrained_vit.pth', weights_only=True), strict=False) 
history_ft = train_and_eval(model_ft, "Finetuned", epochs=20, is_finetuning=True) 

# --- Final Comparison Table ---
print("\n" + "="*30)
print(f"{'Metric':<15} | {'Scratch':<10} | {'Finetuned':<10}")
print("-" * 40)
print(f"{'Final Acc':<15} | {history_scratch['val_acc'][-1]:.2%} | {history_ft['val_acc'][-1]:.2%}")
print(f"{'Final MSE':<15} | {history_scratch['val_mse'][-1]:.4f} | {history_ft['val_mse'][-1]:.4f}")
print("="*30)

# Save the final requested weights
torch.save(model_ft.state_dict(), 'best_linear_vit_final.pth')
print("Weights saved to 'best_linear_vit_final.pth'")

--- Baseline: Training from Scratch ---

Starting Training: Scratch


Scratch - Epoch 1: 100%|██████████| 63/63 [00:27<00:00,  2.26it/s]


[Scratch] Val Acc: 50.75% | Val MSE (Scaled): 0.2990


Scratch - Epoch 2: 100%|██████████| 63/63 [00:13<00:00,  4.63it/s]


[Scratch] Val Acc: 50.75% | Val MSE (Scaled): 0.2966


...

[Finetuned] Val Acc: 78.70% | Val MSE (Scaled): 0.1876


Finetuned - Epoch 19: 100%|██████████| 63/63 [00:12<00:00,  5.19it/s]


[Finetuned] Val Acc: 79.10% | Val MSE (Scaled): 0.1838


Finetuned - Epoch 20: 100%|██████████| 63/63 [00:12<00:00,  5.23it/s]


[Finetuned] Val Acc: 79.45% | Val MSE (Scaled): 0.1774

Metric          | Scratch    | Finetuned 
----------------------------------------
Final Acc       | 83.65% | 79.45%
Final MSE       | 0.1352 | 0.1774
Weights saved to 'best_linear_vit_final.pth'
